In [ ]:
import sys
from pathlib import Path

def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "configs" / "visdrone.yaml").is_file():
            return candidate
    raise FileNotFoundError("Run this notebook from the project directory or one of its subfolders.")

PROJECT_ROOT = find_project_root()
import os
os.chdir(PROJECT_ROOT)

print("Python executable:", sys.executable)

try:
    import torch
    print("PyTorch version:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())

    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except ImportError as e:
    print("PyTorch import failed:", e)

try:
    import ultralytics
    print("Ultralytics version:", ultralytics.__version__)
except ImportError as e:
    print("Ultralytics import failed:", e)
    

# CMPE 401 — Instructor-Defined Project 1

## 1. Project Overview

### 1.1 Objective

This project investigates object detection using YOLOv11n on the VisDrone2019-DET dataset. The objective is to establish a baseline detector, evaluate its performance using quantitative metrics, and investigate whether a controlled modification to the training configuration can improve detection performance.

### 1.2 Research Questions

This project addresses the following questions:

1. How well does the baseline YOLOv11n model detect objects in the VisDrone2019-DET dataset?
2. What do the training and validation loss curves reveal about the model's learning behavior?
3. How does changing a selected training parameter affect detection performance and computational cost?
4. What limitations remain after comparing the baseline and experimental results?

### 1.3 Dataset and Model

- **Dataset:** VisDrone2019-DET
- **Task:** Object detection
- **Baseline model:** YOLOv11n with pretrained weights
- **Number of object classes:** 10
- **Training images:** 6,471
- **Validation images:** 548
- **Baseline input resolution:** 640 × 640
- **Baseline training epochs:** 50

### 1.4 Experimental Approach

The project follows an experimental workflow:

1. Prepare the dataset and convert annotations into YOLO format.
2. Validate the converted labels and dataset structure.
3. Train and evaluate the baseline model.
4. Analyze the training curves and per-class detection performance.
5. Conduct a controlled experiment with a documented change in configuration.
6. Compare quantitative results and discuss the findings, limitations, and potential improvements.

### 1.5 Reproducibility

Training scripts, configuration files, evaluation results, and analysis records are maintained within the project repository. The final repository will include instructions for reproducing the experiments and identifying the trained model checkpoints.



Cell outputs were cleared for a clean, portable copy. The exported plots are in `results/`, and per-epoch logs are in `runs/`.


## 2. Dataset Preparation

### 2.1 Dataset Description

This project uses the VisDrone2019-DET dataset for object detection. The dataset contains images captured from drone perspectives, making it suitable for investigating object detection under challenging viewing conditions.

The dataset is divided into training and validation sets.

| Split | Number of Images | Number of Label Files | Valid Bounding Boxes |
|---|---:|---:|---:|
| Training | 6,471 | 6,471 | 343,204 |
| Validation | 548 | 548 | 38,759 |
| **Total** | **7,019** | **7,019** | **381,963** |

### 2.2 Object Categories

The original annotations were converted into the YOLO detection format. The project uses the following 10 object categories:

1. Pedestrian
2. People
3. Bicycle
4. Car
5. Van
6. Truck
7. Tricycle
8. Awning-tricycle
9. Bus
10. Motor

The category IDs are mapped to zero-based class indices, as required by the YOLO label format.

### 2.3 Annotation Conversion

The original VisDrone annotations were converted into YOLO-format text files. Each valid bounding box is represented by a class index and normalized bounding-box coordinates:

`class_id x_center y_center width height`

The coordinates are normalized relative to the image dimensions.

Annotations belonging to ignored regions or categories excluded

In [ ]:

from pathlib import Path
import random
import cv2
import matplotlib.pyplot as plt

PROJECT_ROOT = find_project_root()
IMAGE_DIR = PROJECT_ROOT / "datasets" / "visdrone_yolo" / "images" / "train"
LABEL_DIR = PROJECT_ROOT / "datasets" / "visdrone_yolo" / "labels" / "train"

CLASS_NAMES = [
    "pedestrian", "people", "bicycle", "car", "van",
    "truck", "tricycle", "awning-tricycle", "bus", "motor"
]

random.seed(42)

image_paths = sorted(IMAGE_DIR.glob("*"))
image_paths = [p for p in image_paths if p.suffix.lower() in {".jpg", ".jpeg", ".png"}]

print("Training images found:", len(image_paths))

if not image_paths:
    raise FileNotFoundError(f"No images found in {IMAGE_DIR}")

selected_images = random.sample(image_paths, min(4, len(image_paths)))

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for ax, image_path in zip(axes, selected_images):
    image = cv2.imread(str(image_path))

    if image is None:
        ax.set_title(f"Could not read: {image_path.name}")
        ax.axis("off")
        continue

    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    height, width = image.shape[:2]

    label_path = LABEL_DIR / f"{image_path.stem}.txt"
    box_count = 0

    if label_path.exists():
        with open(label_path, "r", encoding="utf-8") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) != 5:
                    continue

                class_id = int(parts[0])
                x_center, y_center, box_width, box_height = map(float, parts[1:])

                x1 = int((x_center - box_width / 2) * width)
                y1 = int((y_center - box_height / 2) * height)
                x2 = int((x_center + box_width / 2) * width)
                y2 = int((y_center + box_height / 2) * height)

                cv2.rectangle(image, (x1, y1), (x2, y2), (255, 50, 50), 2)

                if 0 <= class_id < len(CLASS_NAMES):
                    class_name = CLASS_NAMES[class_id]
                else:
                    class_name = f"class_{class_id}"

                cv2.putText(
                    image,
                    class_name,
                    (max(0, x1), max(15, y1 - 5)),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.45,
                    (255, 50, 50),
                    1,
                    cv2.LINE_AA,
                )
                box_count += 1

    ax.imshow(image)
    ax.set_title(f"{image_path.name} | Boxes: {box_count}")
    ax.axis("off")

for ax in axes[len(selected_images):]:
    ax.axis("off")

plt.tight_layout()
plt.show()


### 2.6 Visual Inspection of Annotations

Four randomly selected training images were visualized with their converted YOLO bounding boxes and class labels.

The selected images represent different scenes, including a road under construction, a busy intersection, an urban street, and a residential parking area.

The visualizations show that bounding boxes are distributed across relevant objects and that the converted annotations are not affected by an obvious global coordinate offset. Dense scenes contain many overlapping or closely spaced boxes, reflecting the challenges of object detection in aerial imagery.

This inspection provides an initial qualitative check of the converted labels. However, four images are not sufficient to establish that every annotation is correct. Additional inspection of small objects, class assignments, and bounding-box alignment would provide stronger evidence of annotation quality.



## 3. Baseline Experiment

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display, Image

root = PROJECT_ROOT
baseline = root / "runs" / "yolo11n_baseline"


history = pd.read_csv(baseline / "results.csv")

print("Baseline training history")
print("Recorded epochs:", len(history))
print("Available columns:", list(history.columns))

display(history.tail())


display(Image(filename=str(baseline / "results.png")))


### 3.1 Experimental Setup

The baseline experiment uses the pretrained YOLOv11n model to establish a reference point for evaluating object detection performance on VisDrone2019-DET.

| Parameter | Setting |
|---|---|
| Model | YOLOv11n |
| Initial weights | `yolo11n.pt` |
| Dataset | VisDrone2019-DET |
| Input image size | 640 × 640 |
| Training epochs | 50 |
| Batch size | 8 |
| Device | NVIDIA GeForce RTX 3060 Laptop GPU |
| Random seed | 42 |
| Data configuration | `configs/visdrone.yaml` |
| Training output | `runs/yolo11n_baseline/` |

The model was trained using pretrained weights. The training configuration, evaluation logs, and saved checkpoints were retained for subsequent analysis.

### 3.2 Evaluation Metrics

The baseline model is evaluated using four principal object detection metrics:

- **Precision:** The proportion of predicted positive detections that are correct.
- **Recall:** The proportion of ground-truth objects that are successfully detected.
- **mAP@50:** Mean average precision evaluated at an intersection-over-union threshold of 0.50.
- **mAP@50–95:** Mean average precision averaged across intersection-over-union thresholds from 0.50 to 0.95.

These metrics describe different aspects of detection performance. Precision and recall characterize detection errors, while mAP summarizes precision–recall performance across classes and evaluation thresholds.



### 3.3 Baseline Evaluation Results

The validation results of the selected best checkpoint are summarized below.

| Metric | Value |
|---|---:|
| Precision | 0.437 |
| Recall | 0.329 |
| mAP@50 | 0.314 |
| mAP@50–95 | 0.176 |

The baseline achieves a higher mAP@50 than mAP@50–95. This indicates that the evaluation is more favorable at the less stringent IoU threshold than when localization quality is evaluated across a wider range of thresholds.

The recall of 0.329 also indicates that many ground-truth objects remain undetected under the evaluation settings. These results establish a reference point for subsequent experiments; any proposed improvement must be assessed using comparable evaluation settings.



## 4. Training Curves and Convergence Analysis

This section examines the training and validation losses and the evolution of detection metrics over the 50 training epochs. The objective is to assess the model's learning behavior and identify potential signs of underfitting or overfitting.

In [ ]:

from pathlib import Path
from IPython.display import display, Image

PROJECT_ROOT = find_project_root()
BASELINE_DIR = PROJECT_ROOT / "runs" / "yolo11n_baseline"

curve_files = [
    "results.png",
    "BoxPR_curve.png",
    "BoxF1_curve.png",
    "confusion_matrix.png",
]

for filename in curve_files:
    filepath = BASELINE_DIR / filename
    print(f"\n{filename}")

    if filepath.exists():
        display(Image(filename=str(filepath)))
    else:
        print("File not found:", filepath)


### 4.1 Training Loss and Detection Metrics

The training curves show consistent decreases in box loss, classification loss, and distribution focal loss over the 50 training epochs. This indicates that the model continues to improve its fitting to the training data.

Precision and recall also exhibit an overall upward trend. Precision increases from approximately 0.28 to 0.44, while recall increases from approximately 0.21 to 0.33. Both metrics show some fluctuations during training, but their overall trends are positive.

These observations suggest that the baseline model learns useful object detection features during training. However, training loss and detection metrics alone are insufficient to determine whether the model generalizes well. Validation losses and validation mAP must also be examined before drawing conclusions about convergence or overfitting.

### 4.2 Precision–Recall and F1–Confidence Analysis

The precision–recall curves reveal substantial variation in detection performance across object categories. The car category achieves an AP@50 of approximately 0.729, whereas the bicycle category achieves approximately 0.071. This difference indicates that the baseline model does not perform equally well across all categories.

The overall F1–confidence curve reaches a peak F1 score of approximately 0.36 at a confidence threshold of approximately 0.179 in the displayed evaluation. This illustrates the trade-off between precision and recall as the confidence threshold changes. The selected threshold should be interpreted in the context of the validation procedure rather than as a universally optimal value.

### 4.3 Confusion Matrix Analysis

The confusion matrix provides evidence of prediction errors involving object categories and background-related detections. These errors motivate further investigation of missed objects, false detections, and confusion between visually similar categories.

The displayed matrix is useful for identifying broad error patterns. More detailed numerical conclusions should be based on the underlying evaluation data rather than estimates from the image alone.

### 4.4 Overall Baseline Assessment

The baseline achieves a validation mAP@50 of 0.314 and mAP@50–95 of 0.176. The training and validation loss curves decrease overall, while detection metrics improve over the training period.

However, the model exhibits uneven performance across categories, and the overall recall remains limited. These findings provide a basis for a controlled experiment that evaluates whether a targeted training modification improves detection performance without imposing excessive computational cost.

## 5. Controlled Experiment 1: Input Resolution

### 5.1 Motivation

The baseline model uses an input resolution of 640 × 640. Objects captured from drone perspectives may occupy relatively few pixels, making small-object detection a potential challenge.

Increasing the input resolution may preserve more spatial detail and improve the detection of small objects. However, it also increases computational cost and GPU memory requirements. Therefore, the effect of input resolution must be evaluated experimentally.

### 5.2 Hypothesis

Increasing the input resolution from 640 × 640 to 960 × 960 may improve detection performance, particularly for small objects. This improvement is not guaranteed and must be verified using validation results.

### 5.3 Experimental Design

| Setting | Baseline | Experiment 1 |
|---|---|---|
| Model | YOLOv11n | YOLOv11n |
| Input resolution | 640 × 640 | 960 × 960 |
| Batch size | 8 | 4 |
| Training epochs | 50 | 50 |
| Random seed | 42 | 42 |
| Dataset | VisDrone2019-DET | VisDrone2019-DET |
| Pretrained initialization | `yolo11n.pt` | `yolo11n.pt` |

The experimental batch size is reduced to accommodate the GPU memory constraint. Consequently, input resolution is the primary factor of interest, but batch size is also different and must be acknowledged when interpreting the results.

### 5.4 Evaluation Plan

The experiment will be evaluated using Precision, Recall, mAP@50, and mAP@50–95. Training duration and GPU memory usage will also be recorded.

The experiment will be compared with the baseline to determine whether the change produces a measurable benefit and whether any performance improvement justifies the additional computational cost.



In [ ]:

import torch
import ultralytics

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Ultralytics:", ultralytics.__version__)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory allocated:",
        round(torch.cuda.memory_allocated(0) / 1024**3, 3),
        "GiB"
    )


In [ ]:

from ultralytics import YOLO
from pathlib import Path
import torch
import time

PROJECT_ROOT = find_project_root()
DATA = PROJECT_ROOT / "configs" / "visdrone.yaml"

# Clear unused cached GPU memory before testing.
torch.cuda.empty_cache()

print("Starting the 960-resolution smoke test...")
print("GPU:", torch.cuda.get_device_name(0))

model = YOLO("yolo11n.pt")
start_time = time.time()

try:
    model.train(
        data=str(DATA),
        epochs=1,
        imgsz=960,
        batch=4,
        device=0,
        workers=4,
        project=str(PROJECT_ROOT / "runs"),
        name="yolo11n_exp1_smoketest",
        seed=42,
        plots=False,
        save=False,
        patience=1,
        exist_ok=True,
    )
    print("Smoke test completed successfully.")
except torch.cuda.OutOfMemoryError:
    print("GPU out of memory. The test needs a smaller configuration.")
    raise
finally:
    print(f"Elapsed time: {(time.time() - start_time) / 60:.2f} minutes")
    print(
        "Peak GPU memory:",
        f"{torch.cuda.max_memory_allocated(0) / 1024**3:.2f} GiB"
    )


In [ ]:
from pathlib import Path

run_dir = PROJECT_ROOT / "runs" / "yolo11n_exp1_smoketest"

print("Run directory exists:", run_dir.exists())

for name in ["results.csv", "weights/best.pt", "weights/last.pt"]:
    path = run_dir / name
    print(f"{name}: {path.exists()}")

In [ ]:
import pandas as pd
from pathlib import Path

run_dir = PROJECT_ROOT / "runs" / "yolo11n_exp1_smoketest"
df = pd.read_csv(run_dir / "results.csv")

print("Results shape:", df.shape)
display(df.tail())

In [ ]:
from pathlib import Path

root = PROJECT_ROOT

for name in [
    "yolo11n_baseline",
    "yolo11n_exp1_smoketest",
    "yolo11n_exp1_960",
]:
    path = root / "runs" / name
    print(f"{name}: {'EXISTS' if path.exists() else 'NOT CREATED'}")

In [ ]:

from pathlib import Path
import pandas as pd

root = PROJECT_ROOT
baseline = root / "runs" / "yolo11n_baseline"
smoke = root / "runs" / "yolo11n_exp1_smoketest"
experiment2 = root / "runs" / "yolo11n_exp1_960"

# Experiment 1: verify saved training history
baseline_csv = baseline / "results.csv"
assert baseline_csv.exists(), "Baseline results.csv is missing."

baseline_history = pd.read_csv(baseline_csv)
print("Experiment 1 — Baseline")
print("Recorded epochs:", len(baseline_history))
print("Best model exists:", (baseline / "weights" / "best.pt").exists())

# Smoke test: verify the 960 configuration worked
print("\n960-resolution smoke test")
print("Results CSV exists:", (smoke / "results.csv").exists())
print("Best model exists:", (smoke / "weights" / "best.pt").exists())

# Experiment 2: check that the formal run name is unused
print("\nExperiment 2 — Formal run")
print("Output directory already exists:", experiment2.exists())


In [ ]:

from pathlib import Path
import time
import torch
from ultralytics import YOLO

# Project paths
PROJECT_ROOT = find_project_root()
DATA = PROJECT_ROOT / "configs" / "visdrone.yaml"
RUN_DIR = PROJECT_ROOT / "runs" / "yolo11n_exp1_960"

# Safety checks
assert torch.cuda.is_available(), "CUDA GPU is not available."
assert DATA.exists(), f"Dataset config not found: {DATA}"
assert not RUN_DIR.exists(), (
    f"Run directory already exists: {RUN_DIR}\n"
    "Please check the existing experiment before training again."
)

# Prepare GPU
torch.cuda.empty_cache()


print("GPU:", torch.cuda.get_device_name(0))
print("Model: YOLO11n")
print("Image size: 960")
print("Batch size: 4")
print("Epochs: 50")
print("Output directory:", RUN_DIR)
print("\nStarting Experiment 2...")

# Load pretrained YOLO11n weights
model = YOLO("yolo11n.pt")
start_time = time.time()

# Formal training
results = model.train(
    data=str(DATA),
    epochs=50,
    imgsz=960,
    batch=4,
    device=0,
    workers=2,
    project=str(PROJECT_ROOT / "runs"),
    name="yolo11n_exp1_960",
    seed=42,
    plots=True,
    save=True,
    save_period=10,
    patience=15,
    exist_ok=False,
)

# Report final status
elapsed_minutes = (time.time() - start_time) / 60
peak_memory_gib = torch.cuda.max_memory_allocated(0) / 1024**3

print("\nTraining completed.")
print(f"Elapsed time: {elapsed_minutes:.1f} minutes")
print(f"Peak GPU memory: {peak_memory_gib:.2f} GiB")
print("Results directory:", RUN_DIR)


In [ ]:

from pathlib import Path
import pandas as pd
from IPython.display import display, Image

root = PROJECT_ROOT
exp2 = root / "runs" / "yolo11n_exp1_960"

# Load the 960-resolution training history
history_960 = pd.read_csv(exp2 / "results.csv")

print("Recorded epochs:", len(history_960))
print("\nLast five epochs:")
display(history_960.tail())

# Display the training curves
display(Image(filename=str(exp2 / "results.png")))


### Training Curve Analysis — Experiment 2

The YOLO11n model was trained on the VisDrone dataset at an input resolution of 960 × 960 for 50 epochs, using a batch size of 4.

**Training loss:** The box loss, classification loss, and distribution focal loss decreased throughout training, indicating progressive improvement in object localization and classification.

**Validation loss:** The validation box loss, classification loss, and distribution focal loss also decreased and gradually flattened toward the end of training. No clear divergence between training and validation losses was observed, so the curves do not show an obvious classical overfitting pattern.

**Detection performance:** Precision, recall, mAP@50, and mAP@50–95 improved over the course of training. The final validation metrics were 0.526 precision, 0.402 recall, 0.414 mAP@50, and 0.242 mAP@50–95.

**Comparison and limitations:** The 960-resolution experiment achieved better final validation metrics than the 640-resolution baseline. However, the batch size also changed from 8 to 4, so the improvement cannot be attributed exclusively to input resolution. A controlled experiment with the same batch size would provide stronger evidence about the effect of resolution.

In [ ]:

from pathlib import Path
import pandas as pd

PROJECT_DIR = find_project_root()
RUNS_DIR = PROJECT_DIR / "runs"

experiments = {
    "Baseline (640, batch 8)": RUNS_DIR / "yolo11n_baseline" / "results.csv",
    "Experiment 1 (960, batch 4)": RUNS_DIR / "yolo11n_exp1_960" / "results.csv",
}

rows = []

for experiment_name, csv_path in experiments.items():
    if not csv_path.exists():
        print(f"Missing results file: {csv_path}")
        continue

    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()

    # Find the epoch with the highest validation mAP50-95
    best_idx = df["metrics/mAP50-95(B)"].idxmax()
    best = df.loc[best_idx]

    rows.append({
        "Experiment": experiment_name,
        "Best Epoch": int(best["epoch"]) if "epoch" in df.columns else int(best_idx + 1),
        "Precision": best["metrics/precision(B)"],
        "Recall": best["metrics/recall(B)"],
        "mAP50": best["metrics/mAP50(B)"],
        "mAP50-95": best["metrics/mAP50-95(B)"],
    })

comparison = pd.DataFrame(rows)

if not comparison.empty:
    display(
        comparison.style.format({
            "Precision": "{:.4f}",
            "Recall": "{:.4f}",
            "mAP50": "{:.4f}",
            "mAP50-95": "{:.4f}",
        })
    )
else:
    print("No experiment results were loaded. Check the file paths.")



### Quantitative Comparison: Baseline vs. Experiment 1

The baseline model was trained using an input image size of 640 and a batch size of 8. Experiment 1 used an input image size of 960 and a batch size of 4. Both experiments used YOLO11n and were trained for 50 epochs.

Experiment 1 achieved better results across all four evaluation metrics:

- Precision increased from 0.4384 to 0.5258.
- Recall increased from 0.3282 to 0.4020.
- mAP50 increased from 0.3137 to 0.4138.
- mAP50-95 increased from 0.1756 to 0.2421, representing a relative improvement of approximately 37.9%.

These results suggest that the 960-resolution training configuration improved both detection accuracy and localization performance on the validation dataset.

A possible explanation is that higher-resolution input images preserve more spatial detail, which may help the detector recognize small or densely packed objects in the VisDrone dataset.

However, this experiment changed both image resolution and batch size. Therefore, the observed improvement cannot be attributed to image resolution alone. A follow-up experiment with a batch size of 4 and an image size of 640 would help isolate the effect of image resolution.

Both models achieved their highest validation mAP50-95 at epoch 50. This does not necessarily prove that training had fully converged, so the training curves should also be considered when interpreting the results.


In [ ]:

from pathlib import Path

PROJECT_DIR = find_project_root()
SCRIPT_PATH = PROJECT_DIR / "src" / "train_control_640_b4.py"

script = r'''
from pathlib import Path
from ultralytics import YOLO
import torch

PROJECT_DIR = Path(__file__).resolve().parents[1]
DATA_YAML = PROJECT_DIR / "configs" / "visdrone.yaml"
RUN_DIR = PROJECT_DIR / "runs" / "yolo11n_control_640_b4_nb"

if RUN_DIR.exists():
    raise FileExistsError(
        f"Run directory already exists: {RUN_DIR}\n"
        "Please inspect it before running; do not overwrite existing results."
    )

if not DATA_YAML.exists():
    raise FileNotFoundError(f"Dataset config not found: {DATA_YAML}")

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available. Training stopped.")

model = YOLO("yolo11n.pt")

model.train(
    data=str(DATA_YAML),
    epochs=50,
    imgsz=640,
    batch=4,
    device=0,
    workers=2,
    seed=42,
    plots=True,
    save=True,
    save_period=10,
    patience=15,
    project=str(PROJECT_DIR / "runs"),
    name="yolo11n_control_640_b4_nb",
    exist_ok=False,
)

print("Training completed.")
print("Results saved to:", RUN_DIR)
'''

SCRIPT_PATH.parent.mkdir(parents=True, exist_ok=True)
SCRIPT_PATH.write_text(script.strip() + "\n", encoding="utf-8")

print("Script created:", SCRIPT_PATH)
print("Existing baseline and 960-resolution experiment were not modified.")


In [ ]:

from pathlib import Path
import torch
from ultralytics import YOLO

PROJECT_DIR = find_project_root()
DATA_YAML = PROJECT_DIR / "configs" / "visdrone.yaml"

# Use a NEW directory so existing experiments remain untouched.
RUN_NAME = "yolo11n_control_640_b4_nb"
RUN_DIR = PROJECT_DIR / "runs" / RUN_NAME

print("CUDA available:", torch.cuda.is_available())
print("Dataset config exists:", DATA_YAML.exists())
print("Output directory:", RUN_DIR)

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. Training was not started.")

if not DATA_YAML.exists():
    raise FileNotFoundError(f"Dataset config not found: {DATA_YAML}")

if RUN_DIR.exists():
    raise FileExistsError(
        f"Output directory already exists: {RUN_DIR}. "
        "Choose a different RUN_NAME to protect existing results."
    )

model = YOLO("yolo11n.pt")

results = model.train(
    data=str(DATA_YAML),
    epochs=50,
    imgsz=640,
    batch=4,
    device=0,
    workers=0,
    seed=42,
    plots=True,
    save=True,
    save_period=10,
    patience=15,
    project=str(PROJECT_DIR / "runs"),
    name=RUN_NAME,
    exist_ok=False,
)

print("\nTraining finished.")
print("Results saved to:", RUN_DIR)
print("Best model:", RUN_DIR / "weights" / "best.pt")



## Controlled Experiment: Effect of Input Resolution

### Experimental Design

Three experiments were conducted using the YOLO11n model on the VisDrone dataset:

- Baseline: image size 640, batch size 8.
- Control: image size 640, batch size 4.
- Experiment 1: image size 960, batch size 4.

The control experiment was introduced to reduce the batch-size confound in the initial comparison. The control and Experiment 1 both used batch size 4, 50 training epochs, the same dataset configuration, and random seed 42.

### Results

The control model achieved Precision = 0.426, Recall = 0.325, mAP50 = 0.306, and mAP50-95 = 0.169.

The 960-resolution model achieved Precision = 0.5258, Recall = 0.4020, mAP50 = 0.4138, and mAP50-95 = 0.2421.

Relative to the control model, the 960-resolution model improved:

- Precision by 9.98 percentage points.
- Recall by 7.70 percentage points.
- mAP50 by 10.78 percentage points.
- mAP50-95 by 7.31 percentage points.

### Interpretation

The results support the hypothesis that higher input resolution can improve object detection performance on the VisDrone dataset. Higher-resolution images may preserve more spatial detail for small or densely packed objects.

The comparison between the control and Experiment 1 is more informative than the original baseline comparison because both experiments used the same batch size. However, the results represent a single training run per configuration, and further repetitions would be needed to quantify variability.

### Conclusion

Among the three tested configurations, YOLO11n trained at image size 960 with batch size 4 achieved the strongest validation performance. Its mAP50-95 was 0.2421, compared with 0.169 for the 640-resolution control model.


In [ ]:

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

PROJECT_DIR = find_project_root()
RESULTS_DIR = PROJECT_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Validation metrics obtained from the best checkpoints
comparison_data = [
    {
        "Experiment": "Baseline",
        "Image Size": 640,
        "Batch Size": 8,
        "Precision": 0.4384,
        "Recall": 0.3282,
        "mAP50": 0.3137,
        "mAP50-95": 0.1756,
    },
    {
        "Experiment": "Control",
        "Image Size": 640,
        "Batch Size": 4,
        "Precision": 0.4260,
        "Recall": 0.3250,
        "mAP50": 0.3060,
        "mAP50-95": 0.1690,
    },
    {
        "Experiment": "Resolution 960",
        "Image Size": 960,
        "Batch Size": 4,
        "Precision": 0.5258,
        "Recall": 0.4020,
        "mAP50": 0.4138,
        "mAP50-95": 0.2421,
    },
]

comparison_df = pd.DataFrame(comparison_data)

# Display the table
display(
    comparison_df.style.format({
        "Precision": "{:.4f}",
        "Recall": "{:.4f}",
        "mAP50": "{:.4f}",
        "mAP50-95": "{:.4f}",
    })
)

# Save the table as CSV
csv_path = RESULTS_DIR / "final_experiment_comparison.csv"
comparison_df.to_csv(csv_path, index=False)

# Create and save a metric comparison chart
metrics = ["Precision", "Recall", "mAP50", "mAP50-95"]
ax = comparison_df.set_index("Experiment")[metrics].plot(
    kind="bar",
    figsize=(10, 6),
    ylim=(0, 0.6),
    rot=0,
)

ax.set_title("YOLO11n Validation Performance Comparison")
ax.set_ylabel("Metric Value")
ax.set_xlabel("Experiment")
ax.grid(axis="y", alpha=0.3)
ax.legend(title="Metric")
plt.tight_layout()

chart_path = RESULTS_DIR / "final_experiment_comparison.png"
plt.savefig(chart_path, dpi=200, bbox_inches="tight")
plt.show()

print("\nSaved files:")
print(csv_path)
print(chart_path)


In [ ]:

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_DIR = find_project_root()
RESULTS_DIR = PROJECT_DIR / "results"

experiments = {
    "Baseline (640, batch 8)": PROJECT_DIR / "runs" / "yolo11n_baseline" / "results.csv",
    "Control (640, batch 4)": PROJECT_DIR / "runs" / "yolo11n_control_640_b4_nb" / "results.csv",
    "Resolution 960 (batch 4)": PROJECT_DIR / "runs" / "yolo11n_exp1_960" / "results.csv",
}

plt.figure(figsize=(10, 6))

for name, csv_path in experiments.items():
    if not csv_path.exists():
        print(f"Missing results file: {csv_path}")
        continue

    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()

    # Locate the mAP50-95 validation metric column
    metric_cols = [
        col for col in df.columns
        if "mAP50-95" in col and "metrics/" in col
    ]

    if not metric_cols:
        print(f"Could not find mAP50-95 column in: {csv_path}")
        print("Available columns:", df.columns.tolist())
        continue

    epoch_col = "epoch" if "epoch" in df.columns else df.columns[0]
    metric_col = metric_cols[0]

    plt.plot(
        df[epoch_col] + 1,
        df[metric_col],
        label=name,
        linewidth=2
    )

plt.title("Validation mAP50-95 Across Training Epochs")
plt.xlabel("Epoch")
plt.ylabel("mAP50-95")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()

curve_path = RESULTS_DIR / "training_map5095_comparison.png"
plt.savefig(curve_path, dpi=200, bbox_inches="tight")
plt.show()

print("Training curve saved to:", curve_path)



## Training Curve Analysis

The validation mAP50-95 curves show that all three experiments improved during training, with smaller gains toward the later epochs.

The baseline configuration (640 pixels, batch size 8) increased from approximately 0.085 to 0.176. The control configuration (640 pixels, batch size 4) followed a similar trajectory and finished at approximately 0.169.

The 960-pixel configuration with batch size 4 remained above both 640-pixel configurations throughout the plotted training period. It reached a final mAP50-95 of 0.2421, compared with 0.1690 for the 640-pixel control. This supports the hypothesis that higher input resolution improves detection performance on this VisDrone setup.

The curves became more stable toward the end of training, although small fluctuations remained. Therefore, the results suggest diminishing improvements rather than proving complete convergence. Since this figure displays validation mAP50-95 rather than training and validation losses, it cannot alone establish whether overfitting occurred. Loss curves should be examined separately.


In [ ]:

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_DIR = find_project_root()
RESULTS_DIR = PROJECT_DIR / "results"

experiments = {
    "Baseline (640, batch 8)": PROJECT_DIR / "runs" / "yolo11n_baseline" / "results.csv",
    "Control (640, batch 4)": PROJECT_DIR / "runs" / "yolo11n_control_640_b4_nb" / "results.csv",
    "Resolution 960 (batch 4)": PROJECT_DIR / "runs" / "yolo11n_exp1_960" / "results.csv",
}

loss_types = {
    "box_loss": ("train/box_loss", "val/box_loss"),
    "cls_loss": ("train/cls_loss", "val/cls_loss"),
    "dfl_loss": ("train/dfl_loss", "val/dfl_loss"),
}

# Read all existing result files
data = {}
for name, csv_path in experiments.items():
    if not csv_path.exists():
        print(f"Missing file: {csv_path}")
        continue

    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()
    data[name] = df
    print(f"{name}: {len(df)} epochs loaded")

# Create one plot for each loss type
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (loss_name, columns) in zip(axes, loss_types.items()):
    train_col, val_col = columns

    for name, df in data.items():
        epoch_col = "epoch" if "epoch" in df.columns else df.columns[0]
        epochs = df[epoch_col] + 1

        if train_col in df.columns:
            ax.plot(
                epochs, df[train_col],
                label=f"{name} train",
                linewidth=1.8
            )

        if val_col in df.columns:
            ax.plot(
                epochs, df[val_col],
                label=f"{name} val",
                linestyle="--",
                linewidth=1.8
            )

    ax.set_title(loss_name)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.grid(True, alpha=0.3)

axes[0].legend(fontsize=7)
plt.suptitle("Training and Validation Loss Comparison")
plt.tight_layout()

loss_chart_path = RESULTS_DIR / "training_validation_loss_comparison.png"
plt.savefig(loss_chart_path, dpi=200, bbox_inches="tight")
plt.show()

print("Loss comparison chart saved to:", loss_chart_path)



## Training and Validation Loss Analysis

The training and validation loss curves were examined for all three configurations over 50 epochs.

### Box Loss

Box loss generally decreased throughout training for all configurations, indicating improved bounding-box regression. The 960-resolution experiment also showed a clear decline in both training and validation box loss. The curves became flatter toward the later epochs, suggesting diminishing improvements.

### Classification Loss

Training and validation classification loss generally decreased across the three experiments. The 960-resolution model achieved a lower training classification loss toward the end, although a gap remained between its training and validation curves. This gap indicates that training and validation performance are not identical, but it does not by itself prove overfitting.

### Distribution Focal Loss

DFL loss generally declined during training. The 960-resolution experiment showed a relatively stable validation DFL loss during the later epochs, while its training DFL loss continued to decrease slightly.

### Overall Interpretation

The curves show overall learning progress and diminishing changes toward the end of training. No large, sustained increase in validation loss is apparent in these plots, so there is no strong visual evidence of severe overfitting. However, the remaining gaps between training and validation loss suggest that generalization should be interpreted cautiously.

These loss curves should be considered together with the validation metrics. The 960-resolution configuration achieved the strongest final mAP50 and mAP50-95 among the three tested configurations, but the plots alone do not establish the cause of every performance difference.


### Underfitting, Dataset Size, and Model Capacity

The VisDrone split used here contains 6,471 training images and 548 validation images across 10 classes. Although the annotations provide many bounding boxes, the number of distinct scenes and the small size, overlap, and occlusion of objects still make generalization difficult. YOLO11n is the smallest model in the YOLO11 family, which makes it computationally efficient but gives it less representational capacity than larger variants. This may limit its ability to learn difficult small-object and visually similar categories.

The current results do **not** establish that the whole model is underfitting: training and validation losses decrease over the 50 epochs, validation metrics improve, and the curves flatten without a sustained rise in validation loss. At the same time, the 960-pixel model's recall is 0.402, and its per-class mAP50 ranges from 0.152 for bicycle to 0.812 for car. This uneven performance is evidence of remaining class-specific difficulty, but it is not by itself proof of underfitting; object scale, class frequency, occlusion, and annotation difficulty may also contribute.

Overall, the plots show no clear severe overfitting and do not support a definitive diagnosis of global underfitting. The persistent low scores on difficult classes leave open the possibility that limited model capacity or the fixed training setup constrains performance. A future larger-model comparison or a targeted data/augmentation experiment, with other settings held constant, would help distinguish these explanations.


In [ ]:

import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display

PROJECT_DIR = find_project_root()
RESULTS_DIR = PROJECT_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Per-class mAP50 recorded from each model's best-checkpoint validation
class_data = {
    "Class": [
        "pedestrian", "people", "bicycle", "car", "van",
        "truck", "tricycle", "awning-tricycle", "bus", "motor"
    ],
    "Baseline_640_B8": [
        0.329, 0.251, 0.0706, 0.729, 0.359,
        0.302, 0.196, 0.108, 0.456, 0.337
    ],
    "Control_640_B4": [
        0.318, 0.244, 0.0642, 0.730, 0.350,
        0.285, 0.207, 0.105, 0.427, 0.334
    ],
    "Resolution_960_B4": [
        0.466, 0.365, 0.152, 0.812, 0.454,
        0.368, 0.285, 0.169, 0.582, 0.485
    ],
}

class_df = pd.DataFrame(class_data)

display(
    class_df.style.format({
        "Baseline_640_B8": "{:.3f}",
        "Control_640_B4": "{:.3f}",
        "Resolution_960_B4": "{:.3f}",
    })
)

csv_path = RESULTS_DIR / "per_class_map50_comparison.csv"
class_df.to_csv(csv_path, index=False)

ax = class_df.set_index("Class").plot(
    kind="bar",
    figsize=(13, 6),
    rot=35
)

ax.set_title("Per-Class mAP50 Comparison")
ax.set_xlabel("VisDrone Class")
ax.set_ylabel("mAP50")
ax.set_ylim(0, 1)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()

chart_path = RESULTS_DIR / "per_class_map50_comparison.png"
plt.savefig(chart_path, dpi=200, bbox_inches="tight")
plt.show()

print("Saved table:", csv_path)
print("Saved chart:", chart_path)



## Per-Class Detection Performance Analysis

The per-class mAP50 results show that the 960-pixel configuration outperformed both 640-pixel configurations across all ten VisDrone classes.

The largest absolute improvement was observed for the bus class, whose mAP50 increased from 0.456 in the baseline to 0.582 at 960-pixel resolution. The car class achieved the highest mAP50, increasing from 0.729 to 0.812. Pedestrian, people, van, truck, tricycle, awning-tricycle, and motor also improved.

Bicycle remained a challenging category, with mAP50 increasing from 0.071 in the baseline to 0.152 in the 960-pixel experiment. Although this is a substantial relative improvement, its final absolute performance remains low. This suggests that detection difficulty varies across classes.

The two 640-pixel configurations produced broadly similar results, while the 960-pixel configuration improved every class in this comparison. These findings support the use of higher input resolution for this experiment. However, they do not establish that resolution is the only factor responsible for the improvement, and the results should be interpreted within the tested dataset and training settings.
